In [4]:
import pandas as pd

from dotenv import load_dotenv

from app.data.dto.main.MabuxPortLocodeMap import MabuxPortLocodeMap

load_dotenv()

from app.services.db_service import DbService
sql_db_service = DbService()
await sql_db_service.init_pool()


df = pd.read_excel("./data/mabux_port_id_locode_mapping.xlsx")
len(df)

556

In [ ]:
df.set_index("Mabux ID", inplace=True)
df

In [ ]:
df = df[["Port name", "Country", "Mabux locode", "Real locodes"]]
df = df[~df.index.duplicated(keep="first")]
df = df[df['Real locodes'] != "UNKNOWN"]
df.sort_index()
df = df.reset_index()

In [ ]:
import json
records = []
for r  in  json.loads(df.to_json(orient="records" )):
    records.append(MabuxPortLocodeMap(
        mabux_id=r["Mabux ID"],
        port_name=r["Port name"],
        country_name=r["Country"],
        mabux_locode=r["Mabux locode"],
        real_locode= r["Real locodes"]
    ))


In [ ]:
failed = []

for r in records:
    map_db, err = await sql_db_service.add_sea_port_locode_mabux_id(r)
    if not map_db:
        failed.append(r)
        print(err)


In [ ]:
locode, err = await sql_db_service.get_sea_port_locode_by_mabux_id(664)
locode, err

In [7]:
ids, err = await sql_db_service.get_alternative_mabux_ids("TRVAK")
ids, err

([325, 376, 326, 446, 79], None)